In [1]:
import pandas as pd

# VISEM 원본 운동성 데이터 확인
csv_path = r'C:\Users\neo62\sperm-ai\data\raw\VISEM\visem\visem-dataset\semen_analysis_data.csv'

df = pd.read_csv(csv_path)

print(f"총 참가자 수: {len(df)}명")
print(f"\n컬럼 목록:")
for col in df.columns:
    print(f"  - {col}")
print(f"\n처음 3명 데이터:")
print(df.head(3))

총 참가자 수: 85명

컬럼 목록:
  - ID;Sperm concentration (x10⁶/mL);Total sperm count (x10⁶);Ejaculate volume (mL);Sperm vitality (%);Normal spermatozoa (%);Head defects (%);Midpiece and neck defects (%);Tail defects (%);Cytoplasmic droplet (%);Teratozoospermia index;Progressive motility (%);Non progressive sperm motility (%);Immotile sperm (%);High DNA stainability
  -  HDS (%);DNA fragmentation index
  -  DFI (%)

처음 3명 데이터:
                                      ID;Sperm concentration (x10⁶/mL);Total sperm count (x10⁶);Ejaculate volume (mL);Sperm vitality (%);Normal spermatozoa (%);Head defects (%);Midpiece and neck defects (%);Tail defects (%);Cytoplasmic droplet (%);Teratozoospermia index;Progressive motility (%);Non progressive sperm motility (%);Immotile sperm (%);High DNA stainability  \
1;105 3;363 1;3 5;81;2 0;98 0;11 2;38                                                0;3                                                                                                                    

In [2]:
import pandas as pd

csv_path = r'C:\Users\neo62\sperm-ai\data\raw\VISEM\visem\visem-dataset\semen_analysis_data.csv'

# 세미콜론 구분자로 읽기
df = pd.read_csv(csv_path, sep=';')

print(f"총 참가자 수: {len(df)}명")
print(f"\n컬럼 목록:")
for col in df.columns:
    print(f"  - {col}")
print(f"\n처음 3명 데이터 (운동성 관련):")
motility_cols = ['ID', 'Progressive motility (%)', 
                 'Non progressive sperm motility (%)', 
                 'Immotile sperm (%)']
# 컬럼명이 정확한지 확인 후 출력
available = [c for c in motility_cols if c in df.columns]
print(df[available].head(3))

총 참가자 수: 85명

컬럼 목록:
  - ID
  - Sperm concentration (x10⁶/mL)
  - Total sperm count (x10⁶)
  - Ejaculate volume (mL)
  - Sperm vitality (%)
  - Normal spermatozoa (%)
  - Head defects (%)
  - Midpiece and neck defects (%)
  - Tail defects (%)
  - Cytoplasmic droplet (%)
  - Teratozoospermia index
  - Progressive motility (%)
  - Non progressive sperm motility (%)
  - Immotile sperm (%)
  - High DNA stainability, HDS (%)
  - DNA fragmentation index, DFI (%)

처음 3명 데이터 (운동성 관련):
   ID  Progressive motility (%)  Non progressive sperm motility (%)  \
0   1                        51                                  19   
1   2                        22                                  16   
2   3                        18                                  26   

   Immotile sperm (%)  
0                  30  
1                  62  
2                  56  


In [3]:
import os
import pandas as pd

video_dir = r'C:\Users\neo62\sperm-ai\data\raw\VISEM\visem\visem-dataset\videos'
csv_path = r'C:\Users\neo62\sperm-ai\data\raw\VISEM\visem\visem-dataset\semen_analysis_data.csv'

df = pd.read_csv(csv_path, sep=';')

# 영상 파일 목록에서 ID 추출
video_files = sorted(os.listdir(video_dir))
video_ids = []
for f in video_files:
    pid = f.split('_')[0]  # 파일명 첫 번째 숫자가 ID
    video_ids.append({'filename': f, 'ID': int(pid)})

video_df = pd.DataFrame(video_ids)

# CSV와 매핑
merged = pd.merge(video_df, df[['ID', 'Progressive motility (%)', 
                                 'Non progressive sperm motility (%)', 
                                 'Immotile sperm (%)']], on='ID')

print(f"영상-CSV 매핑 성공: {len(merged)}개")
print(f"\n처음 5개 확인:")
print(merged[['filename', 'ID', 'Progressive motility (%)', 
              'Non progressive sperm motility (%)', 
              'Immotile sperm (%)']].head(5).to_string())

영상-CSV 매핑 성공: 85개

처음 5개 확인:
                         filename  ID  Progressive motility (%)  Non progressive sperm motility (%)  Immotile sperm (%)
0  10_12.03.12_minor drift_HH.avi  10                        26                                  42                  32
1             11_09.01.23_JMA.avi  11                        11                                  17                  72
2             12_09.01.23_SSW.avi  12                        33                                  54                  13
3             13_09.01.26_SSW.avi  13                        33                                  30                  37
4             14_09.01.27_SSW.avi  14                        41                                  43                  16


In [4]:
import os
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from collections import defaultdict
import yaml
import json
from pathlib import Path

# 경로 설정
model = YOLO(r'C:\Users\neo62\sperm-ai\models\yolo11_sperm_v2\weights\best.pt')
video_dir = r'C:\Users\neo62\sperm-ai\data\raw\VISEM\visem\visem-dataset\videos'
csv_path = r'C:\Users\neo62\sperm-ai\data\raw\VISEM\visem\visem-dataset\semen_analysis_data.csv'
config_path = r'C:\Users\neo62\sperm-ai\bytetrack_custom.yaml'
save_path = r'C:\Users\neo62\sperm-ai\outputs\visem_features.json'

# CSV 로드
df_label = pd.read_csv(csv_path, sep=';')

# 영상-CSV 매핑
video_files = sorted(os.listdir(video_dir))
video_map = {}
for f in video_files:
    pid = int(f.split('_')[0])
    video_map[pid] = f

# 이미 처리된 결과 불러오기 (중간 저장 활용)
if os.path.exists(save_path):
    with open(save_path, 'r') as f:
        all_features = json.load(f)
    print(f"기존 저장 데이터 로드: {len(all_features)}개")
else:
    all_features = {}
    print("새로 시작합니다")

def extract_features_visem(video_path):
    """영상에서 운동성 특징 추출"""
    # 전체 정자 수 N (첫 10프레임 중앙값)
    cap = cv2.VideoCapture(video_path)
    counts = []
    for _ in range(10):
        ret, frame = cap.read()
        if not ret:
            break
        res = model(frame, verbose=False, conf=0.3)
        counts.append(int((res[0].boxes.cls == 0).sum()))
    cap.release()

    if not counts:
        return None
    N = int(np.median(counts))

    # ByteTrack 추적
    cap = cv2.VideoCapture(video_path)
    track_history = defaultdict(list)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    max_frames = min(int(fps * 5), total_frames, 250)  # 최대 5초

    for fidx in range(max_frames):
        ret, frame = cap.read()
        if not ret:
            break
        res = model.track(frame, persist=True,
                          tracker=config_path,
                          verbose=False, conf=0.3)
        if res[0].boxes.id is not None:
            for box, tid, cls in zip(
                res[0].boxes.xywh.cpu().numpy(),
                res[0].boxes.id.cpu().numpy().astype(int),
                res[0].boxes.cls.cpu().numpy().astype(int)
            ):
                if cls == 0:
                    track_history[tid].append(
                        (fidx, float(box[0]), float(box[1])))
    cap.release()

    # 특징 계산
    speeds, lins, straight_dists = [], [], []
    for tid, pts in track_history.items():
        if len(pts) < 5:
            continue
        coords = np.array([(cx, cy) for _, cx, cy in pts])
        dists = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
        total_dist = float(np.sum(dists))
        straight_dist = float(np.sqrt(
            (coords[-1][0]-coords[0][0])**2 +
            (coords[-1][1]-coords[0][1])**2))
        avg_speed = total_dist / len(pts)
        linearity = straight_dist / (total_dist + 1e-6)
        speeds.append(avg_speed)
        lins.append(linearity)
        straight_dists.append(straight_dist)

    if not speeds:
        return None

    speeds = np.array(speeds)
    return {
        'N': N,
        'speed_mean':    float(np.mean(speeds)),
        'speed_median':  float(np.median(speeds)),
        'speed_75':      float(np.percentile(speeds, 75)),
        'speed_90':      float(np.percentile(speeds, 90)),
        'lin_mean':      float(np.mean(lins)),
        'lin_75':        float(np.percentile(lins, 75)),
        'straight_mean': float(np.mean(straight_dists)),
        'ratio_fast':    float(np.mean(speeds > 1.5)),
        'ratio_medium':  float(np.mean((speeds > 0.5) & (speeds <= 1.5))),
        'ratio_slow':    float(np.mean(speeds <= 0.5)),
        'n_tracks':      len(speeds),
    }

# 85개 영상 처리
total = len(video_map)
processed = 0

for pid, filename in sorted(video_map.items()):
    # 이미 처리된 것은 스킵
    if str(pid) in all_features:
        processed += 1
        continue

    # 운동성 CSV에 없는 참가자 스킵
    row = df_label[df_label['ID'] == pid]
    if len(row) == 0:
        continue

    video_path = os.path.join(video_dir, filename)
    print(f"[{processed+1}/{total}] 참가자 {pid} 처리 중... ({filename})")

    feat = extract_features_visem(video_path)

    if feat:
        feat['prog'] = float(row['Progressive motility (%)'].values[0])
        feat['non_prog'] = float(row['Non progressive sperm motility (%)'].values[0])
        feat['immotile'] = float(row['Immotile sperm (%)'].values[0])
        all_features[str(pid)] = feat

        # 10명마다 중간 저장
        if len(all_features) % 10 == 0:
            with open(save_path, 'w') as f:
                json.dump(all_features, f)
            print(f"  → 중간 저장 완료 ({len(all_features)}명)")

    processed += 1

# 최종 저장
with open(save_path, 'w') as f:
    json.dump(all_features, f)

print(f"\n✅ 전체 완료: {len(all_features)}명")
print(f"저장 위치: {save_path}")

새로 시작합니다
[1/85] 참가자 1 처리 중... (1_09.09.02_SSW.avi)
[2/85] 참가자 2 처리 중... (2_09.09.03_lots of debris_SSW.avi)
[3/85] 참가자 3 처리 중... (3_11.01.21_JMA.avi)
[4/85] 참가자 4 처리 중... (4_11.03.29_HH.avi)
[5/85] 참가자 5 처리 중... (5_11.05.04_JMA.avi)
[6/85] 참가자 6 처리 중... (6_12.02.08_drift_HH.avi)
[7/85] 참가자 7 처리 중... (7_12.02.20_minor drift_HH.avi)
[8/85] 참가자 8 처리 중... (8_12.02.21_minor drift_HH.avi)
[9/85] 참가자 9 처리 중... (9_12.03.06_minor drift_HH.avi)
[10/85] 참가자 10 처리 중... (10_12.03.12_minor drift_HH.avi)
  → 중간 저장 완료 (10명)
[11/85] 참가자 11 처리 중... (11_09.01.23_JMA.avi)
[12/85] 참가자 12 처리 중... (12_09.01.23_SSW.avi)
[13/85] 참가자 13 처리 중... (13_09.01.26_SSW.avi)
[14/85] 참가자 14 처리 중... (14_09.01.27_SSW.avi)
[15/85] 참가자 15 처리 중... (15_09.01.28_SSW.avi)
[16/85] 참가자 16 처리 중... (16_09.01.28_SSW.avi)
[17/85] 참가자 17 처리 중... (17_09.01.29_SSW.avi)
[18/85] 참가자 18 처리 중... (18_09.01.29_SSW.avi)
[19/85] 참가자 19 처리 중... (19_10.01.06_JMA.avi)
[20/85] 참가자 20 처리 중... (20_09.02.02_JMA.avi)
  → 중간 저장 완료 (20명)
[21/85] 참가자 21 처리

In [5]:
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import cross_val_score, KFold
import warnings
warnings.filterwarnings('ignore')

# 데이터 로드
save_path = r'C:\Users\neo62\sperm-ai\outputs\visem_features.json'
with open(save_path, 'r') as f:
    all_features = json.load(f)

print(f"총 데이터: {len(all_features)}명")

# 특징/정답 분리
feature_cols = [
    'speed_mean', 'speed_median', 'speed_75', 'speed_90',
    'lin_mean', 'lin_75', 'straight_mean',
    'ratio_fast', 'ratio_medium', 'ratio_slow', 'n_tracks'
]

X, y, pids = [], [], []
for pid, feat in all_features.items():
    X.append([feat[c] for c in feature_cols])
    y.append([feat['prog'], feat['non_prog'], feat['immotile']])
    pids.append(int(pid))

X = np.array(X)
y = np.array(y)

print(f"특징 행렬: {X.shape}")
print(f"정답 행렬: {y.shape}")

# 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── 교차검증으로 실제 성능 측정 ──────────────────────────
print("\n=== 5-Fold 교차검증 (진짜 성능) ===")
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for model_name, reg in [
    ('Ridge', MultiOutputRegressor(Ridge(alpha=1.0))),
    ('SVR',   MultiOutputRegressor(SVR(kernel='rbf', C=1.0)))
]:
    mae_prog, mae_non, mae_imm = [], [], []
    for train_idx, val_idx in kf.split(X_scaled):
        X_tr, X_val = X_scaled[train_idx], X_scaled[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        reg.fit(X_tr, y_tr)
        y_pred = reg.predict(X_val)
        mae_prog.append(np.mean(np.abs(y_pred[:,0] - y_val[:,0])))
        mae_non.append(np.mean(np.abs(y_pred[:,1] - y_val[:,1])))
        mae_imm.append(np.mean(np.abs(y_pred[:,2] - y_val[:,2])))

    avg = (np.mean(mae_prog)+np.mean(mae_non)+np.mean(mae_imm))/3
    print(f"\n[{model_name}]")
    print(f"  전진 운동성 MAE:   {np.mean(mae_prog):.1f}%p")
    print(f"  비전진 운동성 MAE: {np.mean(mae_non):.1f}%p")
    print(f"  비운동성 MAE:      {np.mean(mae_imm):.1f}%p")
    print(f"  전체 평균:         {avg:.1f}%p")

print(f"\n[이전 결과 (16명 Ridge)]")
print(f"  전체 평균: 15.1%p")

총 데이터: 85명
특징 행렬: (85, 11)
정답 행렬: (85, 3)

=== 5-Fold 교차검증 (진짜 성능) ===

[Ridge]
  전진 운동성 MAE:   8.4%p
  비전진 운동성 MAE: 6.1%p
  비운동성 MAE:      7.5%p
  전체 평균:         7.3%p

[SVR]
  전진 운동성 MAE:   13.7%p
  비전진 운동성 MAE: 7.1%p
  비운동성 MAE:      10.6%p
  전체 평균:         10.4%p

[이전 결과 (16명 Ridge)]
  전체 평균: 15.1%p


In [6]:
import pickle

# 최종 모델 전체 데이터로 재학습
final_model = MultiOutputRegressor(Ridge(alpha=1.0))
final_model.fit(X_scaled, y)

# 저장
model_save = {
    'model': final_model,
    'scaler': scaler,
    'feature_cols': feature_cols
}

with open(r'C:\Users\neo62\sperm-ai\models\motility_regressor.pkl', 'wb') as f:
    pickle.dump(model_save, f)

print("✅ 최종 회귀 모델 저장 완료")
print(f"경로: models/motility_regressor.pkl")
print(f"\n최종 성능 (5-Fold CV):")
print(f"  전진 운동성 MAE:   8.4%p")
print(f"  비전진 운동성 MAE: 6.1%p")
print(f"  비운동성 MAE:      7.5%p")
print(f"  전체 평균:         7.3%p ← 논문 최고 수준과 동등!")

✅ 최종 회귀 모델 저장 완료
경로: models/motility_regressor.pkl

최종 성능 (5-Fold CV):
  전진 운동성 MAE:   8.4%p
  비전진 운동성 MAE: 6.1%p
  비운동성 MAE:      7.5%p
  전체 평균:         7.3%p ← 논문 최고 수준과 동등!
